# Task 2.2 – Real-time Violation Visualisation

This notebook implements the visualisation component of the AWAS-style traffic monitoring system.  
It reads violation records that have already been detected by the Spark Structured Streaming application and stored in MongoDB, then updates the dashboard repeatedly to support real-time monitoring.

The purpose of this visualisation is not only to display raw violation data, but also to help traffic operators answer operational questions such as:

- Are violations increasing or decreasing in the most recent period?
- When do violation counts reach their maximum or minimum?
- Are there sudden spikes or drops in violation speed?
- What speed values are unusually high compared with the current distribution?
- Is the system continuing to receive new violation records from MongoDB?

The assignment specification requires visual outputs that show violation counts and speed patterns over arrival time, with labelled and technically interpretable plots. It also requires notable points such as maxima, minima, peaks, or sudden changes to be marked and explained operationally.

## Visualisation Workflow

The visualisation follows this workflow:

1. Connect to the MongoDB `traffic_monitoring` database.
2. Read records from the `violations` collection.
3. Use MongoDB aggregation pipelines to extract nested violation events from each daily vehicle document.
4. Poll the database every two seconds to detect newly inserted violation records.
5. Maintain two rolling visualisation windows:
   - a short 60-second window for real-time violation count monitoring;
   - a longer 5-minute window for speed distribution and anomaly analysis.
6. Update the dashboard in Jupyter using Matplotlib and `ipywidgets.Output`.
7. Annotate important points such as maximum values, minimum values, speed spikes, speed drops, and dynamic percentile thresholds.

The visualisation is designed to run while the streaming application is writing detected violations into MongoDB.

## End-to-End Visualisation Flow

The visualisation notebook reads violation records that were produced by the Spark Structured Streaming application and stored in MongoDB. It then applies two different aggregation windows to support real-time operational monitoring.

```text
Spark Streaming Application
        |
        v
MongoDB violations collection
        |
        v
Aggregation Pipeline 1
Short 60-second count and speed window
        |
        v
Subplot A: real-time count + average speed

MongoDB violations collection
        |
        v
Aggregation Pipeline 2
Long 5-minute speed distribution window
        |
        v
Subplot B: percentile-based speed anomaly analysis
```

## Key Parameters

| Parameter | Value | Purpose |
|---|---:|---|
| `HOST_IP` | `host.docker.internal` | Connects the notebook to the MongoDB container from Jupyter/Docker environment. |
| `POLL_INTERVAL` | `2 seconds` | Controls how often the visualisation checks MongoDB for newly inserted violations. |
| `WINDOW_SIZE_SHORT` | `60 seconds` | Defines the rolling time window for Subplot A, which focuses on recent violation counts and average speed. |
| `WINDOW_SIZE_LONG` | `300 seconds / 5 minutes` | Defines the rolling time window for Subplot B, which focuses on recent speed distribution and anomaly detection. |
| MongoDB database | `traffic_monitoring` | Stores the processed traffic-monitoring data. |
| MongoDB collection | `violations` | Stores detected instantaneous and average-speed violations. |
| Timestamp field | `violations.inserted_at` | Used as the real-time processing timestamp for visualisation. |
| Speed field | `violations.speed_reading` | Used to plot violation speed patterns and calculate dynamic statistics. |
| Short-window aggregation | last 2 seconds | Counts newly inserted violations in each polling interval. |
| Long-window aggregation | last 5 minutes | Groups recent violation speeds by second and analyses their distribution. |
| Moving average window | last 10 seconds | Smooths short-term speed fluctuation in Subplot B. |
| Percentile thresholds | 25th, 50th, 75th, 90th, 95th | Dynamically classify low, medium, high, and very high violation speeds. |

## MongoDB Query Design

The visualisation reads from the `violations` collection.  
The streaming application stores multiple violations for the same vehicle and date inside an embedded `violations` array. Therefore, this notebook uses `$unwind` to flatten the embedded violation array before calculating counts and speed statistics.

Two aggregation pipelines are used.

### Pipeline for Subplot A

Subplot A only needs newly inserted violations from the most recent polling interval.  
The query filters documents by `last_updated`, unwinds the embedded `violations` array, and then filters individual violation entries using `violations.inserted_at`.

This prevents old violations inside an updated daily document from being counted again.

### Pipeline for Subplot B

Subplot B analyses the last five minutes of violation speed records.  
It groups violation records by second and calculates the average speed for each second. This provides a clearer real-time speed trend than plotting every raw record individually, especially when many records arrive close together.

## Subplot A – Real-time Violations and Average Speed

Subplot A shows two metrics over a rolling 60-second window:

1. The number of new violations detected every 2 seconds.
2. The average speed of the newly detected violations.

A dual-axis line chart is used because the two metrics have different units:

- the left y-axis shows violation count;
- the right y-axis shows average speed in km/h.

This plot supports operational monitoring because it shows whether the system is currently detecting many new violations and whether those violations are associated with high speeds.

The chart marks the maximum and minimum values in the visible window.  
The maximum violation count helps identify the busiest or most dangerous recent interval.  
The maximum average speed helps identify the period with the most severe speeding behaviour.

## Subplot B – Real-time Speed Distribution and Dynamic Analysis

Subplot B shows violation average speeds over a rolling 5-minute window.

Instead of only plotting a simple line chart, this subplot dynamically computes percentiles from the current window:

- below 50th percentile: low relative speed group;
- 50th to 75th percentile: medium relative speed group;
- 75th to 90th percentile: high relative speed group;
- above 90th percentile: very high relative speed group.

These percentile thresholds are recalculated every time the dashboard updates.  
This makes the visualisation adaptive because the definition of “high” speed is based on the current traffic behaviour rather than a fixed static value.

A 10-second moving average line is also plotted to smooth short-term fluctuation.  
This makes it easier to see whether the overall speed trend is rising, falling, or remaining stable.

## Interesting Point Annotation

The notebook marks several types of interesting points.

### Maximum and Minimum Points

Subplot A labels the maximum and minimum violation count and average speed values in the rolling 60-second window.

These labels are useful because they quickly show the most active and least active moments in the current monitoring period.

### Dynamic Percentile Lines

Subplot B draws horizontal reference lines for the 25th, 50th, 75th, 90th, and 95th percentiles.

These percentile lines provide more insight than a single maximum or minimum point because they show the full distribution of recent violation speeds.  
For example, a violation above the 90th percentile represents one of the highest-speed violations in the current 5-minute monitoring window.

### Spike Annotation

A speed spike is annotated when the maximum speed is above the 90th percentile and sufficiently higher than the median.  
This highlights unusually severe speeding behaviour compared with the current traffic pattern.

Operationally, this may indicate a dangerous speeding event that should receive priority attention.

### Drop Annotation

A speed drop is annotated when the minimum speed is below the 25th percentile and sufficiently lower than the median.  
This can indicate a sudden change in traffic behaviour, such as congestion, braking, or a reduction in violation severity.

Operationally, this helps distinguish between consistently high-speed violation periods and periods where the speed pattern changes sharply.

## Why This Supports Operational Decisions

This dashboard supports operational decision-making in several ways.

First, the short-window chart provides immediate feedback on whether the streaming system is actively receiving and processing violations.

Second, the violation count trend helps identify when violation activity increases sharply. This can support short-term enforcement decisions.

Third, the speed trend and moving average help identify whether speeding behaviour is becoming more severe over time.

Fourth, the percentile-based classification provides a dynamic way to distinguish normal violation speeds from unusually high violation speeds within the current window.

Finally, the annotated spikes and drops help operators focus on meaningful changes rather than manually inspecting all data points.

## Assumptions and Limitations

This notebook assumes that the Spark streaming application has already detected violations correctly and written them into MongoDB.

It also assumes that each violation entry contains a valid `inserted_at` timestamp and a numeric `speed_reading`.

The visualisation uses `violations.inserted_at` as the real-time processing time. This is suitable for monitoring newly inserted records, but it may differ from the original camera event timestamp.

The dashboard currently focuses on overall violation counts and speed patterns. It does not separately plot instantaneous violations and average-speed violations, and it does not break down results by individual camera or road segment. These could be added as future extensions.

## Future Extensions

Possible future improvements include:

- separating instantaneous and average-speed violations into different lines;
- adding camera-level violation counts;
- adding segment-level average-speed violation counts;
- adding an interactive map view using camera coordinates;
- adding dropdown filters for violation type, camera ID, or date;
- exporting detected peak periods for further analysis.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta, timezone
import numpy as np
import statistics
import time
from pymongo import MongoClient
from IPython.display import display, clear_output
import ipywidgets as widgets

# --- Global Configurations ---
HOST_IP = "host.docker.internal"
POLL_INTERVAL = 2       # Poll the database every 2 seconds

# Independent rolling time window configurations for each subplot
WINDOW_SIZE_SHORT = 60   # Subplot 1 (Line chart): Rolling display for the last 60 seconds
WINDOW_SIZE_LONG = 300   # Subplot 2 (Scatter plot): Rolling display for the last 5 minutes

def connect_mongo():
    """
    Establishes a connection to the MongoDB instance.

    Returns:
        MongoClient or None: A MongoClient instance if successful, None otherwise.
    """
    try: 
        return MongoClient(host=HOST_IP, port=27017)
    except Exception as e: 
        print('[ERROR] Connection failed:', e)
        return None

def draw_annotations_sub1(ax, x_data, y_data, color, is_speed=False):
    """
    Draws bubble annotations for the maximum and minimum values in Subplot 1.

    Args:
        ax (matplotlib.axes.Axes): The targeted subplot axes instance.
        x_data (list): A list of datetime objects representing the x-axis timeline.
        y_data (list): A list of numerical values representing the y-axis metrics.
        color (str): The color theme configuration for the text border.
        is_speed (bool, optional): If True, formats annotations with speed units (km/h). 
            Defaults to False.
    """
    if not y_data: return
    max_idx = np.argmax(y_data)
    min_idx = np.argmin(y_data)
    bbox_props = dict(boxstyle="round,pad=0.3", fc="lightblue" if not is_speed else "thistle", ec=color, lw=1, alpha=0.7)
    
    unit = " km/h" if is_speed else ""
    ax.text(x_data[max_idx], y_data[max_idx], f"MAX: {y_data[max_idx]:.1f}{unit}" if is_speed else f"MAX: {y_data[max_idx]}",
            color='darkblue' if not is_speed else 'indigo', weight='bold', fontsize=7, bbox=bbox_props, ha='center', va='bottom')
    if len(y_data) > 1 and max_idx != min_idx:
        ax.text(x_data[min_idx], y_data[min_idx], f"MIN: {y_data[min_idx]:.1f}{unit}" if is_speed else f"MIN: {y_data[min_idx]}",
                color='darkblue' if not is_speed else 'indigo', weight='bold', fontsize=7, bbox=bbox_props, ha='center', va='top')

def run_combined_dashboard():
    """
    Executes the main pipeline loop for the real-time stream processing dashboard.
    
    Fetches historical aggregation pipelines from MongoDB periodically, structures the data 
    into memory-managed rolling windows, dynamically analyzes quantiles and statistical anomalies, 
    and updates the composite dashboard utilizing anti-flicker Jupyter output components.
    """
    client = connect_mongo()
    if client is None: return
    collection = client['traffic_monitoring']['violations']
    
    # Anti-flicker display output container specialized for Jupyter environments
    out = widgets.Output()
    display(out)
    
    # Initialize a composite figure layout with a 2-row, 1-column subplot structural hierarchy
    fig, (ax_top_left, ax_bottom) = plt.subplots(2, 1, figsize=(11, 9))
    # Allocate coordinate spacing margins to accommodate secondary twin Y-axis layouts
    fig.subplots_adjust(right=0.88, hspace=0.35, bottom=0.12, top=0.92)
    
    # Secondary independent y-axis mapping specifically assigned to Subplot 1 (Average Speed)
    ax_top_right = ax_top_left.twinx()
    
    # --- State Metrics Queue ---
    sub1_timeline_x = []
    sub1_counts_y = []
    sub1_speeds_y = []
    sub1_last_valid_speed = 0.0

    print("Integrated data stream visualization initialized (Top: 60s insights | Bottom: 5min analytics dashboard)...")

    try:
        while True:
            current_time = datetime.now(timezone.utc)
            sub1_start_time = current_time - timedelta(seconds=WINDOW_SIZE_SHORT)
            sub2_start_time = current_time - timedelta(seconds=WINDOW_SIZE_LONG)
            
            current_local_now = datetime.now()
            
            # =================================================================
            # Data Ingestion Phase 1: Aggregate incremental data for Subplot 1 (60s Window)
            # =================================================================
            last_check_time = current_time - timedelta(seconds=POLL_INTERVAL)
            pipeline_sub1 = [
                { '$match': { 'last_updated': { '$gt': last_check_time, '$lte': current_time } } },
                { '$unwind': '$violations' },
                { '$match': { 'violations.inserted_at': { '$gt': last_check_time, '$lte': current_time } } },
                { '$project': { '_id': 0, 'speed': '$violations.speed_reading' } }
            ]
            
            inc_docs_sub1 = list(collection.aggregate(pipeline_sub1))
            v_count_sub1 = len(inc_docs_sub1)
            
            if v_count_sub1 > 0:
                valid_speeds_sub1 = [d['speed'] for d in inc_docs_sub1 if d['speed'] > 0]
                v_speed_avg_sub1 = statistics.mean(valid_speeds_sub1) if valid_speeds_sub1 else sub1_last_valid_speed
                sub1_last_valid_speed = v_speed_avg_sub1
            else:
                v_speed_avg_sub1 = sub1_last_valid_speed
                
            sub1_timeline_x.append(current_local_now)
            sub1_counts_y.append(v_count_sub1)
            sub1_speeds_y.append(v_speed_avg_sub1)
            
            # Bound queue memory leaks by enforcing the 60 seconds rolling layout threshold
            cutoff_sub1 = current_local_now - timedelta(seconds=WINDOW_SIZE_SHORT)
            while sub1_timeline_x and sub1_timeline_x[0] < cutoff_sub1:
                sub1_timeline_x.pop(0)
                sub1_counts_y.pop(0)
                sub1_speeds_y.pop(0)

            # =================================================================
            # Data Ingestion Phase 2: Aggregate historical distribution for Subplot 2 (5min Window)
            # =================================================================
            pipeline_sub2 = [
                { '$match': { 'last_updated': { '$gt': sub2_start_time, '$lte': current_time } } },
                { '$unwind': '$violations' },
                { '$match': { 'violations.inserted_at': { '$gt': sub2_start_time, '$lte': current_time } } },
                {
                    '$group': {
                        '_id': {
                            'year': { '$year': '$violations.inserted_at' },
                            'month': { '$month': '$violations.inserted_at' },
                            'day': { '$dayOfMonth': '$violations.inserted_at' },
                            'hour': { '$hour': '$violations.inserted_at' },
                            'minute': { '$minute': '$violations.inserted_at' },
                            'second': { '$second': '$violations.inserted_at' }
                        },
                        'sec_avg_speed': { '$avg': '$violations.speed_reading' },
                        'exact_ts': { '$first': '$violations.inserted_at' }
                    }
                },
                { '$sort': { 'exact_ts': 1 } }
            ]
            
            docs_sub2 = list(collection.aggregate(pipeline_sub2))
            
            sub2_window_times = []
            sub2_window_speeds = []
            
            for d in docs_sub2:
                if d['sec_avg_speed'] > 0:
                    db_ts_aware = d['exact_ts'].replace(tzinfo=timezone.utc)
                    time_offset = current_local_now - (current_time - db_ts_aware)
                    time_sec_key = time_offset.replace(microsecond=0)
                    
                    sub2_window_times.append(time_sec_key)
                    sub2_window_speeds.append(d['sec_avg_speed'])

            # =================================================================
            # Canvas Rendering & Aesthetic Styling Phase
            # =================================================================
            with out:
                ax_top_left.clear()
                ax_top_right.clear()
                ax_bottom.clear()
                
                # ----------------- Subplot 1 Configurations & Canvas Drawing -----------------
                ax_top_left.set_ylabel('New violations per 2 s', color='blue', fontsize=9, fontweight='bold')
                ax_top_left.tick_params(axis='y', labelcolor='blue')
                ax_top_left.grid(True, linestyle=':', alpha=0.4)
                
                ax_top_right.set_ylabel('Average Speed (km/h)', color='purple', fontsize=9, fontweight='bold')
                ax_top_right.tick_params(axis='y', labelcolor='purple')
                
                ax_top_left.set_title('Subplot A: Real-time Violations & Average Speed (Rolling 60s Window)', fontsize=10, fontweight='bold', loc='left')
                
                if len(sub1_timeline_x) > 1:
                    ax_top_left.plot(sub1_timeline_x, sub1_counts_y, marker='o', linestyle='-', color='blue', linewidth=1.2, label='Violations Count')
                    ax_top_right.plot(sub1_timeline_x, sub1_speeds_y, marker='s', linestyle='-', color='purple', linewidth=1.2, label='Avg Speed')
                    
                    ax_top_left.set_xlim(min(sub1_timeline_x), max(sub1_timeline_x))
                    draw_annotations_sub1(ax_top_left, sub1_timeline_x, sub1_counts_y, 'blue', is_speed=False)
                    draw_annotations_sub1(ax_top_right, sub1_timeline_x, sub1_speeds_y, 'purple', is_speed=True)
                    
                    lines_l, labels_l = ax_top_left.get_legend_handles_labels()
                    lines_r, labels_r = ax_top_right.get_legend_handles_labels()
                    ax_top_left.legend(lines_l + lines_r, labels_l + labels_r, loc='upper left', fontsize=8)
                
                ax_top_left.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
                ax_top_left.set_ylim(0, max(sub1_counts_y) + 5 if sub1_counts_y else 20)
                ax_top_right.set_ylim(0, 200)

                # ----------------- Subplot 2 Configurations & Canvas Drawing -----------------
                ax_bottom.grid(True, linestyle=':', alpha=0.4)
                ax_bottom.set_xlabel('Real-time Processing Time', fontweight='bold', fontsize=9, labelpad=5)
                ax_bottom.set_ylabel('Average Speed (km/h)', fontweight='bold', fontsize=9)
                ax_bottom.set_title('Subplot B: Real-time Violation Average Speeds with Dynamic Analysis (Rolling 5min Window)', fontsize=10, fontweight='bold', loc='left')
                
                if len(sub2_window_speeds) >= 1:
                    speeds_arr = np.array(sub2_window_speeds)
                    times_arr = np.array(sub2_window_times)
                    
                    # A. Compute dynamic window percentile metrics
                    p25, p50, p75, p90, p95 = np.percentile(speeds_arr, [25, 50, 75, 90, 95])
                    
                    # B. Segmentation mappings according to data classification
                    c_very_high = speeds_arr >= p90
                    c_high = (speeds_arr >= p75) & (speeds_arr < p90)
                    c_medium = (speeds_arr >= p50) & (speeds_arr < p75)
                    c_low = speeds_arr < p50
                    
                    # Scattered dots
                    ax_bottom.scatter(times_arr[c_very_high], speeds_arr[c_very_high], color='red', s=25, label='Very High (>=90th)', zorder=4)
                    ax_bottom.scatter(times_arr[c_high], speeds_arr[c_high], color='orange', s=25, label='High (75th-90th)', zorder=4)
                    ax_bottom.scatter(times_arr[c_medium], speeds_arr[c_medium], color='yellow', s=25, label='Medium (50th-75th)', zorder=4)
                    ax_bottom.scatter(times_arr[c_low], speeds_arr[c_low], color='green', s=25, label='Low (<50th)', zorder=4)
                    
                    # C. Mathematical calculation for the sliding 10-second rolling moving average
                    ten_sec_ma_speeds = []
                    for t in sub2_window_times:
                        # Find all vehicle speed data within the current second-level timestamp [t - 10 seconds, t].
                        valid_points_in_10s = [
                            sub2_window_speeds[i] 
                            for i, idx_time in enumerate(sub2_window_times) 
                            if t - timedelta(seconds=10) <= idx_time <= t
                        ]
                        ten_sec_ma_speeds.append(statistics.mean(valid_points_in_10s) if valid_points_in_10s else sub2_window_speeds[sub2_window_times.index(t)])

                    ax_bottom.plot(times_arr, ten_sec_ma_speeds, color='purple', linewidth=2, linestyle='-', label='Moving Average', zorder=5)
                    
                    # D. Lateral dynamic reference quantile
                    x_min_s2, x_max_s2 = current_local_now - timedelta(seconds=WINDOW_SIZE_LONG), current_local_now
                    ax_bottom.set_xlim(x_min_s2, x_max_s2)
                    
                    # Project statistical boundary reference rules onto the chart layout
                    lines = [(p95, 'red', f'95th: {p95:.1f}'), (p90, 'orange', f'90th: {p90:.1f}'), 
                             (p75, '#F4D03F', f'75th: {p75:.1f}'), (p50, 'blue', f'Median: {p50:.1f}'), 
                             (p25, 'green', f'25th: {p25:.1f}')]
                    for val, col, label_text in lines:
                        ax_bottom.axhline(y=val, color=col, linestyle='--', linewidth=1.0, alpha=0.6)
                        ax_bottom.text(x_max_s2 + timedelta(seconds=2), val, label_text, color=col, weight='bold', fontsize=7, va='center')
                    
                    # E. Anomaly marker (adapted to a 5-minute width timedelta offset)
                    max_idx_s2, min_idx_s2 = np.argmax(speeds_arr), np.argmin(speeds_arr)
                    if len(speeds_arr) >= 2:
                        if speeds_arr[max_idx_s2] >= p90 and speeds_arr[max_idx_s2] > p50 * 1.12:
                            pct_above = ((speeds_arr[max_idx_s2] - p50) / p50) * 100
                            ax_bottom.annotate(f'SPIKE!\n{speeds_arr[max_idx_s2]:.1f}\n({pct_above:.1f}% up)',
                                                xy=(times_arr[max_idx_s2], speeds_arr[max_idx_s2]), 
                                                xytext=(times_arr[max_idx_s2] - timedelta(seconds=25), speeds_arr[max_idx_s2] + 12),
                                                bbox=dict(boxstyle="round,pad=0.2", fc="yellow", ec="red", lw=0.8, alpha=0.7),
                                                arrowprops=dict(arrowstyle="->", color="red", lw=1.0), color="red", weight="bold", fontsize=7, ha='center')
                        
                        if speeds_arr[min_idx_s2] <= p25 and speeds_arr[min_idx_s2] < p50 * 0.85:
                            pct_below = ((p50 - speeds_arr[min_idx_s2]) / p50) * 100
                            ax_bottom.annotate(f'DROP!\n{speeds_arr[min_idx_s2]:.1f}\n({pct_below:.1f}% down)',
                                                xy=(times_arr[min_idx_s2], speeds_arr[min_idx_s2]), 
                                                xytext=(times_arr[min_idx_s2] + timedelta(seconds=20), speeds_arr[min_idx_s2] - 12),
                                                bbox=dict(boxstyle="round,pad=0.2", fc="lightblue", ec="blue", lw=0.8, alpha=0.7),
                                                arrowprops=dict(arrowstyle="->", color="blue", lw=1.0), color="blue", weight="bold", fontsize=7, ha='center')
                    
                    # 1. Position runtime statistics text safely in the lower-left corner region
                    v_mean_s2 = statistics.mean(sub2_window_speeds)
                    stats_str = f"Stats: Active Window Secs: {len(sub2_window_speeds)} | 5min Mean: {v_mean_s2:.1f}"
                    ax_bottom.text(0.01, 0.04, stats_str, transform=ax_bottom.transAxes, fontsize=7.5, weight='bold',
                                   bbox=dict(boxstyle="round,pad=0.2", fc="#E5E7E9", ec="gray", alpha=0.7))
                    
                    # 2. Anchor map legends cleanly inside the bottom right canvas corner to avoid overlaps
                    ax_bottom.legend(loc='lower right', fontsize=7, ncol=2)
                
                ax_bottom.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
                ax_bottom.set_ylim(0, 200)
                
                # Unified Beautification and Chart Output
                fig.suptitle(f"Real-time Integrated Traffic Monitoring Control Center\n[Current Time: {current_local_now.strftime('%Y-%m-%d %H:%M:%S')}]", fontsize=12, fontweight='bold')
                fig.autofmt_xdate()
                
                clear_output(wait=True)
                display(fig)
                
            time.sleep(POLL_INTERVAL)

    except KeyboardInterrupt:
        print('\n The large-screen monitoring system has been safely shut down.')
    finally:
        client.close()
        plt.close(fig)

if __name__ == '__main__':
    run_combined_dashboard()